In [6]:
import os
from PIL import Image
import shutil

In [8]:
ROOT_DIR = ""
OUTPUT_DIR = "kitti_yolo"
SPLIT_DIR = "training/splits"
CLASS_MAP = {
    'Car': 0,
    'Pedestrian': 1,
    'Cyclist': 2,
    'Van': 0,
    'Person_sitting': 1,
    'Truck': 0,
}

In [11]:
def convert_to_yolo(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    return (x * dw, y * dh, w * dw, h * dh)

def process_split(split_name):
    os.makedirs(f"{OUTPUT_DIR}/images/{split_name}", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/labels/{split_name}", exist_ok=True)

    with open(f"{SPLIT_DIR}/{split_name}.txt", 'r') as f:
        ids = [line.strip() for line in f.readlines()]

    for img_id in ids:
        # 1. Copy Image
        img_src = f"training/image_2/{img_id}.png"
        img_dst = f"{OUTPUT_DIR}/images/{split_name}/{img_id}.png"
        shutil.copy(img_src, img_dst)

        # 2. Convert Labels
        img = Image.open(img_src)
        w, h = img.size

        with open(f"training/label_2/{img_id}.txt", 'r') as f_in:
            with open(f"{OUTPUT_DIR}/labels/{split_name}/{img_id}.txt", 'w') as f_out:
                for line in f_in:
                    parts = line.split()
                    cls_name = parts[0]
                    if cls_name not in CLASS_MAP: continue

                    cls_id = CLASS_MAP[cls_name]
                    # KITTI format uses x1, y1, x2, y2 at indices 4, 5, 6, 7
                    box = [float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])]
                    yolo_box = convert_to_yolo((w, h), box)
                    f_out.write(f"{cls_id} {' '.join([f'{x:.6f}' for x in yolo_box])}\n")

if __name__ == "__main__":
    process_split("train")
    process_split("val")
    print("Dataset Ready!")

Dataset Ready!


In [12]:
from ultralytics import YOLO

model = YOLO('yolo26s.pt')

model.train(
    data='kitti.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='MuSGD',
    project='KITTI_Object_Detection',
    name='yolo26_kitti_experiment',
    device=0,
    save = True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/pouya/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.34 🚀 Python-3.14.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5771MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgs

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f0d72cce900>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04